# Exact solution used in MES runs


We would like to MES the implementation of cauchy boundary conditions on the opposite side of the sheath enterance

For a field $f$, we have

$$
f(0) = a\\
\partial_z f\big|_0 = b
$$

In [1]:
%matplotlib notebook

import os
import sys

from boutdata.mms import DDX, t, x, y, z
from sympy import S, cos, exp, init_printing, log, pi, sin, sqrt, tanh

# If we add to sys.path, then it must be an absolute path
common_dir = os.path.abspath("./../../../../common")
# Sys path is a list of system paths
sys.path.append(common_dir)
from CELMAPy.MES import BOUT_print, make_plot

init_printing()

## Initialize

In [2]:
folder = "../gaussianWSinAndParabola/"

## Define the variables

In [3]:
# Initialization
the_vars = {}

In [4]:
# We need Lx
from boututils.options import BOUTOptions

myOpts = BOUTOptions(folder)
Lx = eval(myOpts.geom["Lx"])
Ly = eval(myOpts.geom["Ly"])

In [5]:
# Gaussian with sinus and parabola

# The skew sinus
# In cartesian coordinates we would like a sinus with with a wave-vector in the direction
# 45 degrees with respect to the first quadrant. This can be achieved with a wave vector
# k = [1/sqrt(2), 1/sqrt(2)]
# sin((1/sqrt(2))*(x + y))
# We would like 2 nodes, so we may write
# sin((1/sqrt(2))*(x + y)*(2*pi/(2*Lx)))

# The gaussian
# In cartesian coordinates we would like
# f = exp(-(1/(2*w^2))*((x-x0)^2 + (y-y0)^2))

# A parabola
# In cartesian coordinates, we have
# ((x-x0)/Lx)^2

wx = 0.5 * Lx
wy = 0.5 * Ly
x0 = 0.3 * Lx
y0 = 0.5 * Ly
the_vars["f"] = (
    sin((1 / sqrt(2)) * (x + y) * (2 * pi / (2 * Lx)))
    * exp(-(((x - x0) ** 2 / (2 * wx**2)) + ((y - y0) ** 2 / (2 * wy**2))))
    * ((x - x0) / Lx) ** 2
)

the_vars["a"] = the_vars["f"].subs(y, 0)
# The stencil takes b per coord->dy = Ly/ny, while y here runs over [0, 2pi]
the_vars["b"] = the_vars["f"].diff(y).subs(y, 0) * 2 * pi / Ly

## Plot

In [6]:
make_plot(
    folder=folder, the_vars=the_vars, plot2d=True, include_aux=False, direction="y"
)

## Print the variables in BOUT++ format

In [7]:
BOUT_print(the_vars, rational=False)